In [0]:
%run ./00_config

✅ Config loaded : billing.sales_data
📅 Target date   : 2026-10-03


In [0]:
import requests
import base64
import binascii
import re
import io
import uuid
from datetime import datetime, timezone

import pandas as pd
import numpy as np
from pyspark.sql import functions as F
from pyspark.sql.functions import current_timestamp, lit, col
from pyspark.sql.utils import AnalysisException
from pyspark.sql.types import DateType, StringType, TimestampType


# =============================================================================
# CONFIGURATION
# =============================================================================
# These variables are expected to already be defined by your Databricks job:
# API_BASE_URL
# API_USERNAME
# API_PASSWORD
# TARGET_DATE
# catalog_name
# schema_name

TABLE_PATH = f"{catalog_name}.{schema_name}.billing_bronze"
PIPELINE_RUN_ID = str(uuid.uuid4())


# =============================================================================
# API
# =============================================================================
def authenticate():
    print("Authenticating...")

    resp = requests.post(
        f"{API_BASE_URL}/login",
        json={"username": API_USERNAME, "password": API_PASSWORD},
        timeout=10,
    )

    if resp.status_code in (200, 201):
        try:
            token = resp.json().get("token")
        except ValueError as exc:
            raise RuntimeError(
                f"Auth succeeded with HTTP {resp.status_code}, but response was not valid JSON"
            ) from exc

        if token:
            print("✅ Auth successful")
            return token

    raise RuntimeError(
        f"Auth failed: HTTP {resp.status_code} — {resp.text[:1000]}"
    )


def download_csv(token, order_type="online", from_date=None, to_date=None):
    print(
        f"Downloading {order_type} orders: "
        f"{from_date} → {to_date}..."
    )

    resp = requests.get(
        f"{API_BASE_URL}/orders/orderReportCSV",
        params={
            "orderType": order_type,
            "fromDate": from_date,
            "toDate": to_date,
        },
        headers={
            "accept": "*/*",
            "Authorization": token,
        },
        timeout=30,
    )

    if resp.status_code != 200:
        raise RuntimeError(
            f"Download failed: HTTP {resp.status_code} — {resp.text[:1000]}"
        )

    if not resp.content:
        raise RuntimeError("Download returned an empty HTTP response")

    try:
        text = resp.content.decode("utf-8")
    except UnicodeDecodeError as exc:
        raise RuntimeError("Cannot decode response as UTF-8") from exc

    clean = re.sub(r"\s", "", text)

    try:
        csv_data = base64.b64decode(clean, validate=True).decode("utf-8")
    except (binascii.Error, ValueError, UnicodeDecodeError):
        # API may return raw CSV rather than base64.
        csv_data = text

    try:
        df = pd.read_csv(io.StringIO(csv_data), low_memory=False)
    except Exception as exc:
        raise RuntimeError(
            f"Downloaded response could not be parsed as CSV: {exc}"
        ) from exc

    print(f"✅ Downloaded {len(df):,} rows")
    return df


# =============================================================================
# VALIDATION
# =============================================================================
def validate_not_empty(df, stage):
    if df is None:
        raise RuntimeError(f"❌ None dataframe at {stage} — aborting pipeline")

    if hasattr(df, "empty") and df.empty:
        raise RuntimeError(f"❌ Empty dataframe at {stage} — aborting pipeline")

    count = len(df) if hasattr(df, "__len__") else df.count()

    if count == 0:
        raise RuntimeError(f"❌ Zero rows at {stage} — aborting pipeline")

    return count


def validate_target_date(target_date):
    try:
        parsed = pd.to_datetime(target_date, format="%Y-%m-%d", errors="raise")
    except Exception as exc:
        raise ValueError(
            f"TARGET_DATE must be YYYY-MM-DD; received {target_date!r}"
        ) from exc

    return parsed.strftime("%Y-%m-%d")


def validate_source_columns(df: pd.DataFrame, existing_schema):
    expected = {
        field.name
        for field in existing_schema.fields
        if field.name not in ("ingested_at", "source_date")
    }

    actual = set(df.columns)
    missing = sorted(expected - actual)

    if missing:
        raise RuntimeError(
            "API schema validation failed. Missing columns required by the "
            f"existing Bronze schema: {missing}"
        )


# =============================================================================
# PANDAS → BRONZE TYPE NORMALIZATION
# =============================================================================
def clean_pandas_for_bronze(df: pd.DataFrame, existing_schema) -> pd.DataFrame:
    """
    Normalize API data to the existing Bronze Delta schema before Spark conversion.

    This intentionally keeps type coercion in pandas so Spark does not have to
    infer incompatible API types such as scientific-notation IDs or mixed numeric
    strings.
    """
    from pyspark.sql.types import LongType, IntegerType, DoubleType, FloatType

    df = df.copy()
    df.columns = df.columns.astype(str).str.strip()

    for field in existing_schema.fields:
        if field.name not in df.columns:
            continue

        c = field.name

        if isinstance(field.dataType, (LongType, IntegerType)):
            numeric = pd.to_numeric(df[c], errors="coerce")

            # Reject non-integral numeric values instead of silently truncating.
            non_integral = numeric.notna() & (numeric % 1 != 0)
            if non_integral.any():
                sample = df.loc[non_integral, c].head(5).tolist()
                raise RuntimeError(
                    f"Column {c!r} contains non-integral values for an integer "
                    f"Bronze field. Examples: {sample}"
                )

            df[c] = numeric.apply(
                lambda x: int(x) if pd.notna(x) else None
            )

        elif isinstance(field.dataType, (DoubleType, FloatType)):
            df[c] = pd.to_numeric(df[c], errors="coerce")

        else:
            # Preserve strings while converting NaN/NaT to None.
            df[c] = df[c].where(pd.notna(df[c]), None)
            df[c] = df[c].astype(object)
            df[c] = df[c].apply(
                lambda x: str(x) if x is not None else None
            )

    return df


# =============================================================================
# TABLE HELPERS
# =============================================================================
def table_exists(table_name):
    try:
        return bool(spark.catalog.tableExists(table_name))
    except Exception as exc:
        raise RuntimeError(
            f"Could not determine whether Bronze table exists: {table_name}"
        ) from exc


def get_existing_schema(table_name):
    try:
        return spark.table(table_name).schema
    except AnalysisException as exc:
        raise RuntimeError(
            f"Bronze table {table_name} exists but its schema could not be read"
        ) from exc


# =============================================================================
# ATOMIC DELTA WRITE
# =============================================================================
def atomic_replace_date(bronze_df, table_name, target_date):
    """
    Atomically replace the Delta rows belonging to one source_date.

    The existing Bronze table is the schema contract. In particular, older
    versions of this pipeline may have created source_date as STRING, while
    newer versions prefer DATE. We therefore conform source_date to the
    existing Delta type before replaceWhere so Delta never attempts to merge
    STRING and DATE fields.
    """
    target_schema = spark.table(table_name).schema
    source_date_field = next(
        (field for field in target_schema.fields if field.name == "source_date"),
        None,
    )

    if source_date_field is None:
        raise RuntimeError(
            f"Existing Bronze table {table_name} has no source_date column"
        )

    target_type = source_date_field.dataType

    if isinstance(target_type, DateType):
        write_df = bronze_df.withColumn(
            "source_date", F.to_date(F.lit(target_date))
        )
        predicate = f"source_date = DATE '{target_date}'"

    elif isinstance(target_type, StringType):
        write_df = bronze_df.withColumn(
            "source_date", F.lit(target_date).cast("string")
        )
        predicate = f"source_date = '{target_date}'"

        print(
            "⚠️ Existing Bronze source_date is STRING. "
            "Keeping that schema for compatibility. "
            "A future one-time schema migration can convert it to DATE."
        )

    elif isinstance(target_type, TimestampType):
        write_df = bronze_df.withColumn(
            "source_date",
            F.to_timestamp(F.lit(f"{target_date} 00:00:00")),
        )
        predicate = f"source_date = TIMESTAMP '{target_date} 00:00:00'"

    else:
        raise RuntimeError(
            "Unsupported source_date type in existing Bronze table: "
            f"{target_type.simpleString()}"
        )

    # Re-select the target schema order after the type correction.
    target_columns = [field.name for field in target_schema.fields]
    write_df = write_df.select(target_columns)

    print(f"🔄 Atomically replacing Bronze snapshot for {target_date}")
    print(f"   Existing source_date type: {target_type.simpleString()}")
    print(f"   replaceWhere: {predicate}")

    (
        write_df.write
        .format("delta")
        .mode("overwrite")
        .option("replaceWhere", predicate)
        .saveAsTable(table_name)
    )

    print("✅ Atomic Delta replacement committed")


# =============================================================================
# CREATE INITIAL TABLE
# =============================================================================
def create_initial_bronze(bronze_df, table_name):
    print(f"ℹ️  Bronze table does not exist — creating {table_name}")

    (
        bronze_df.write
        .format("delta")
        .mode("overwrite")
        .partitionBy("source_date")
        .saveAsTable(table_name)
    )

    print("✅ Initial Bronze table created")


# =============================================================================
# POST-WRITE VALIDATION
# =============================================================================
def validate_written_snapshot(table_name, target_date, expected_count, run_id):
    target_schema = spark.table(table_name).schema
    source_date_field = next(
        (field for field in target_schema.fields if field.name == "source_date"),
        None,
    )

    if source_date_field is None:
        raise RuntimeError(
            f"Bronze table {table_name} has no source_date column after write"
        )

    target_type = source_date_field.dataType

    if isinstance(target_type, DateType):
        target_value = F.to_date(F.lit(target_date))
    elif isinstance(target_type, StringType):
        target_value = F.lit(target_date).cast("string")
    elif isinstance(target_type, TimestampType):
        target_value = F.to_timestamp(F.lit(f"{target_date} 00:00:00"))
    else:
        raise RuntimeError(
            "Unsupported source_date type during validation: "
            f"{target_type.simpleString()}"
        )

    actual_df = spark.table(table_name).filter(
        F.col("source_date") == target_value
    )

    actual_count = actual_df.count()

    print(
        f"Bronze validation: expected={expected_count:,}, "
        f"actual={actual_count:,}"
    )

    if actual_count == 0:
        raise RuntimeError(
            f"❌ Bronze write completed but 0 rows exist for {target_date}"
        )

    if actual_count != expected_count:
        raise RuntimeError(
            f"❌ Bronze row-count validation failed for {target_date}: "
            f"expected {expected_count:,}, found {actual_count:,}"
        )

    print(
        f"✅ Bronze validation passed for {target_date} "
        f"(run_id={run_id})"
    )

    return actual_count


# =============================================================================
# RUN
# =============================================================================
TARGET_DATE = validate_target_date(TARGET_DATE)

print("=" * 80)
print("PRODUCTION BILLING BRONZE INGESTION")
print(f"Pipeline run ID : {PIPELINE_RUN_ID}")
print(f"Target date     : {TARGET_DATE}")
print(f"Target table    : {TABLE_PATH}")
print(f"Started at      : {datetime.now(timezone.utc).isoformat()}")
print("=" * 80)

try:
    # -------------------------------------------------------------------------
    # 1. Authenticate
    # -------------------------------------------------------------------------
    token = authenticate()

    # -------------------------------------------------------------------------
    # 2. Download source snapshot
    # -------------------------------------------------------------------------
    df_raw = download_csv(
        token,
        order_type="online",
        from_date=TARGET_DATE,
        to_date=TARGET_DATE,
    )

    api_count = validate_not_empty(df_raw, "API download")

    # -------------------------------------------------------------------------
    # 3. Determine whether this is an existing-table replacement or first load
    # -------------------------------------------------------------------------
    exists = table_exists(TABLE_PATH)

    if exists:
        existing_schema = get_existing_schema(TABLE_PATH)

        # Validate the API shape against the established Bronze contract.
        df_raw.columns = df_raw.columns.astype(str).str.strip()
        validate_source_columns(df_raw, existing_schema)

        # ---------------------------------------------------------------------
        # 4. Normalize API values to the existing Delta schema
        # ---------------------------------------------------------------------
        df_clean = clean_pandas_for_bronze(df_raw, existing_schema)

        # ---------------------------------------------------------------------
        # 5. Convert to Spark
        # ---------------------------------------------------------------------
        df_spark = spark.createDataFrame(df_clean)

        existing_cols = [field.name for field in existing_schema.fields]

        source_cols = [
            c for c in existing_cols
            if c not in ("ingested_at", "source_date")
        ]

        # Select only columns belonging to the established Bronze schema.
        select_cols = [col(c) for c in source_cols]

        bronze_df = (
            df_spark
            .select(select_cols)
            .withColumn("ingested_at", current_timestamp())
            .withColumn("source_date", F.to_date(F.lit(TARGET_DATE)))
            .select(existing_cols)
        )

        # Validate before mutating Delta.
        spark_count = bronze_df.count()
        if spark_count != api_count:
            raise RuntimeError(
                "Pre-write row-count changed during Spark conversion: "
                f"API={api_count:,}, Spark={spark_count:,}"
            )

        if spark_count == 0:
            raise RuntimeError("Pre-write Spark dataframe contains 0 rows")

        # ---------------------------------------------------------------------
        # 6. ATOMIC replacement — no DELETE + INSERT
        # ---------------------------------------------------------------------
        atomic_replace_date(
            bronze_df,
            TABLE_PATH,
            TARGET_DATE,
        )

    else:
        # ---------------------------------------------------------------------
        # First-ever creation: there is no existing schema to conform to.
        # Infer the initial schema once, then partition by source_date.
        # ---------------------------------------------------------------------
        df_raw.columns = df_raw.columns.astype(str).str.strip()
        df_initial = df_raw.where(pd.notna(df_raw), None)

        bronze_df = (
            spark.createDataFrame(df_initial)
            .withColumn("ingested_at", current_timestamp())
            .withColumn("source_date", F.to_date(F.lit(TARGET_DATE)))
        )

        initial_count = bronze_df.count()
        if initial_count != api_count:
            raise RuntimeError(
                "Pre-write row-count changed during initial Spark conversion: "
                f"API={api_count:,}, Spark={initial_count:,}"
            )

        if initial_count == 0:
            raise RuntimeError("Initial Bronze dataframe contains 0 rows")

        create_initial_bronze(bronze_df, TABLE_PATH)

    # -------------------------------------------------------------------------
    # 7. Post-write verification
    # -------------------------------------------------------------------------
    final_count = validate_written_snapshot(
        TABLE_PATH,
        TARGET_DATE,
        api_count,
        PIPELINE_RUN_ID,
    )

    print("=" * 80)
    print("✅ BRONZE INGESTION COMPLETED SUCCESSFULLY")
    print(f"Pipeline run ID : {PIPELINE_RUN_ID}")
    print(f"Target date     : {TARGET_DATE}")
    print(f"Rows written    : {final_count:,}")
    print(f"Completed at    : {datetime.now(timezone.utc).isoformat()}")
    print("=" * 80)

except Exception as exc:
    print("=" * 80)
    print("❌ BILLING BRONZE INGESTION FAILED")
    print(f"Pipeline run ID : {PIPELINE_RUN_ID}")
    print(f"Target date     : {TARGET_DATE}")
    print(f"Error           : {type(exc).__name__}: {exc}")
    print(f"Failed at       : {datetime.now(timezone.utc).isoformat()}")
    print("=" * 80)
    raise


PRODUCTION BILLING BRONZE INGESTION
Pipeline run ID : 03ccf202-4238-4b38-9cda-59da9e53a073
Target date     : 2026-10-03
Target table    : billing.sales_data.billing_bronze
Started at      : 2026-10-04T10:12:15.445288+00:00
Authenticating...
✅ Auth successful
✅ Downloaded 52,995 rows
⚠️ Existing Bronze source_date is STRING. Keeping that schema for compatibility. A future one-time schema migration can convert it to DATE.
🔄 Atomically replacing Bronze snapshot for 2026-10-03
   Existing source_date type: string
   replaceWhere: source_date = '2026-10-03'
✅ Atomic Delta replacement committed
Bronze validation: expected=52,995, actual=52,995
✅ Bronze validation passed for 2026-10-03 (run_id=03ccf202-4238-4b38-9cda-59da9e53a073)
✅ BRONZE INGESTION COMPLETED SUCCESSFULLY
Pipeline run ID : 03ccf202-4238-4b38-9cda-59da9e53a073
Target date     : 2026-10-03
Rows written    : 52,995
Completed at    : 2026-10-04T10:12:47.554308+00:00
